# 第11回：化学の知識を特徴量にする

**今日の問い：研究者の知識を、モデルへ渡せる形にするにはどうするか。**

**セルの動かし方**：各セル（灰色の枠）を選んで `Shift + Enter`（またはセル左の▷ボタン）を押すと実行できます。
**上から順に**実行してください。前のセルを飛ばすと、後のセルでエラーになります。

`TRY`は全員、`CHANGE`は値を1つ変える練習、`CHALLENGE`は余裕がある人向けです。
`DEEP DIVE`・`APPENDIX`は経験者や自習向けの発展で、飛ばしても本編は完結します。
分からないコードは、セル全体ではなく気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
# 【準備セル】教材フォルダの場所を自動で見つけます。中身は今は理解しなくてOK、そのまま実行してください。
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回でできるようになること

- 化学的仮説を再計算可能な特徴量へ変え、交差検証でアブレーションする
- リークを避けたtarget encodingを、分割の内側で自作する
- 相互情報量・RFECVで特徴量を選び、適用領域の限界を意識する

### 進み方

`CORE`は同期90分で扱う本線、`DEEP DIVE`は時間があれば扱う深掘り、
`SELF-STUDY`は任意自習です。すべて終わらなくても次回へ進めます。
経験者は`CORE`を早めに終え、`DEEP DIVE`を5人で分担して読むと深まります。

### 先に押さえる言葉

- 特徴量設計：既存情報から予測に役立つ表現を作ること
- target encoding：カテゴリを目的変数の集約値で置き換える手法
- アブレーション：要素を足し引きして寄与を調べる比較
- RFECV：交差検証つきで再帰的に特徴量を削る選択法
- 適用領域：モデルが信頼できる入力範囲

> **実行前の30秒予想**：今日の問いに、今の言葉で仮の答えを書いてから始めます。


## 特徴量設計＝あなたの化学知識をモデルへ渡す

モデルは与えられた列しか見ません。**「最適温度から離れるほど収率が落ちる」**という知識を持っていても、
`temperature_c`の生の値だけでは、モデルがその山型を学ぶのは大変です。そこで、知識を**計算式**にして
新しい列（特徴量）として渡します。これが特徴量設計です。

鉄則が2つあります。
1. **予測時点で計算できること**（第5回。実験後の値から作らない）。
2. **追加の効果は、同じ検証条件で前後比較して確かめる**（思い込みで良し悪しを決めない）。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


## TRY：仮説を計算式にする

2つの仮説を式にします。**「最適温度78℃からの距離」**（離れるほど収率減、という山型を直接表す）と、
**「単位時間あたりの濃度」**（濃度と時間の兼ね合い）。どちらも計画時に計算できる値です。


In [ ]:
engineered = df.copy()
engineered["temperature_distance"] = (engineered["temperature_c"] - 78).abs()
engineered["concentration_per_hour"] = engineered["concentration_m"] / engineered["reaction_time_h"]
engineered[["temperature_c", "temperature_distance", "concentration_per_hour"]].head()


### 読みどころ

`temperature_distance`は、78℃から上下どちらに離れても大きくなる値（絶対値）。第4回で見た「温度と収率の
山型」を、モデルにとって学びやすい**単調な形**に翻訳しています。生の温度より効くかどうかは、次で検証します。


## アブレーション：追加の効果を「同じ条件」で確かめる

**アブレーション**とは、要素を足し引きして寄与を測る比較のこと。特徴量を追加する前後で、
**同じモデル・同じ交差検証**でMAEを比べます。これをやらずに「良さそうだから採用」は禁物です。


In [ ]:
from sklearn.model_selection import cross_val_score, KFold
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

base = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa"]
added = [*base, "temperature_distance", "concentration_per_hour"]
cv = KFold(5, shuffle=True, random_state=42)
for label, cols in {"追加前": base, "追加後": added}.items():
    est = make_pipeline(SimpleImputer(strategy="median"), RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42))
    scores = cross_val_score(est, engineered[cols], engineered["yield_pct"], cv=cv, scoring="neg_mean_absolute_error")
    print(f"{label}: MAE={-scores.mean():.3f} ± {scores.std():.3f}")


### 出力の読み方

- 「追加後」のMAEが「追加前」より**下がっていれば**、その特徴量は効いています。**ばらつき(±)より大きく**下がっているかも見ます（±の中の差は誤差かも）。
- 効かない・悪化することもあります。それも立派な結果——**「この仮説はこのモデルには効かなかった」**と分かるのが検証の価値です。悪化した実験も記録します（第12回）。


## CORE深掘り：関連の強い特徴量を選ぶ（相互情報量）

特徴量が増えると、効かない列がノイズになることも。**相互情報量（第4回）**で目的変数との関連が強い順に
並べ、上位k個を選びます。相関と違い、山型のような非線形の関連も拾えます。


In [ ]:
from functools import partial
from sklearn.feature_selection import SelectKBest, mutual_info_regression

# random_stateを固定しないとMIの推定値は実行ごとに変わる（第4回と同じ作法）
mi_score = partial(mutual_info_regression, random_state=42)
sel_data = engineered[added].fillna(engineered[added].median())
selector = SelectKBest(mi_score, k=4).fit(sel_data, engineered["yield_pct"])
pd.DataFrame({"特徴量": added, "MIスコア": selector.scores_, "選択": selector.get_support()}).sort_values("MIスコア", ascending=False).round(3)


### 出力の読み方（結果は素直に受け止める）

MIスコアの高い順に並び、上位4つに「選択=True」が付きます。ここで大事なのは、**自作の`temperature_distance`が
必ず上位に来るとは限らない**ことです。実際、このデータの単変量MIでは上位に来ないことがあります。相互情報量は
**1列ずつ単独で**目的変数との関連を測るため、「他の列と組み合わせて効く」種類の特徴量を低く見積もることがあります。
思い込みで良し悪しを決めず、数字を見る。そして次のDEEP DIVE/APPENDIXで、**別の見方（並べ替え重要度）だと結論が
変わる**ことを実際に確かめます。`random_state`を固定しているのは、固定しないとMIの推定値が毎回変わるためです。


## CHALLENGE：RDKitでSMILESから記述子を計算する

分子量やLogPは、本来は分子構造（SMILES）から計算できます。RDKitが入っていれば、エタノールの
記述子を実際に計算してみます。無い環境では自動でスキップし、計算済みの列で本編を進められます。


In [ ]:
try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, Crippen
    molecule = Chem.MolFromSmiles("CCO")
    print("エタノールの分子量:", round(Descriptors.MolWt(molecule), 2))
    print("エタノールのLogP:", round(Crippen.MolLogP(molecule), 2))
except ImportError:
    print("RDKitは任意（uv sync --extra chemistry）。計算済みmolecular_weight/logp/tpsaで本編を進められます。")


### 読みどころ

RDKitが動けば、SMILES（`CCO`＝エタノール）から分子量やLogPが再現されます。**「記述子＝構造から計算できる
特徴量」**だと腹落ちします。RDKitは発展扱いなので、無くても計算済みの列で全く問題ありません。


## DEEP DIVE：リークしやすい特徴量と、賢い選択

強力だが**リークしやすい**特徴量の代表が**target encoding**（カテゴリを目的変数の平均で置き換える）です。
やり方を誤ると、第6回で学んだリークを自ら仕込むことになります。安全なやり方を身につけます。


### target encoding：全データ平均は「リーク」、OOFなら安全

「系列ごとの平均収率」を特徴量にしたいとします。**全データの平均**で作ると、各行の答えが自分の特徴量に
混ざりリークします。正しくは、第6回の交差検証と同じ発想で、**その行を含まない分割の平均**で作ります
（OOF＝out-of-fold）。両者でMAEを比べ、リークが楽観を生むことを確かめます。


In [ ]:
import numpy as np
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import Ridge

def oof_target_encode(frame, col, target, n_splits=5, seed=42):
    "分割の内側で平均を学習するリーク安全なtarget encoding。"
    encoded = pd.Series(index=frame.index, dtype=float)
    global_mean = frame[target].mean()
    for tr, va in KFold(n_splits, shuffle=True, random_state=seed).split(frame):
        means = frame.iloc[tr].groupby(col)[target].mean()
        encoded.iloc[va] = frame.iloc[va][col].map(means).fillna(global_mean).to_numpy()
    return encoded

leaky = df["scaffold_group"].map(df.groupby("scaffold_group")["yield_pct"].mean())
safe = oof_target_encode(df, "scaffold_group", "yield_pct")
num_cols = ["temperature_c", "concentration_m", "logp"]
X_num = df[num_cols].fillna(df[num_cols].median())
for label, enc in {"リークあり(全データ平均)": leaky, "OOF(安全)": safe}.items():
    feats = X_num.assign(scaffold_te=enc.to_numpy())
    scores = cross_val_score(Ridge(), feats, df["yield_pct"], cv=5, scoring="neg_mean_absolute_error")
    print(f"{label}: MAE={-scores.mean():.3f}")
print("リークありは楽観的に見えることがある。実運用の性能はOOFに近い。")


### 出力の読み方

「リークあり」のMAEが「OOF」より**小さく（良く）見える**ことがあります。しかしそれは幻——本番では
その行の答えは手に入りません。**実運用の実力はOFFの側**。強力な特徴量ほど、作り方のリークに注意します。


### RFECV：交差検証つきで特徴量を絞り込む

`RFECV`は、重要度の低い特徴量を1つずつ削りながら交差検証し、**性能が最も良くなる特徴量の組**を
自動で選びます。人手の取捨選択より客観的です。ここでは**係数の大きさで重要度を測る線形モデル(Ridge)**で
回します（後述のとおり、木モデルはノイズに強すぎてRFECVが列を削らないことが多いため）。尺度をそろえてから
かけます。


In [ ]:
import numpy as np
from sklearn.feature_selection import RFECV
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

# わざと「無意味な列」を混ぜて、RFECVがそれを削れるかを確かめる
rng = np.random.default_rng(0)
rfe_data = engineered[added].fillna(engineered[added].median()).copy()
rfe_data["noise"] = rng.normal(size=len(rfe_data))     # 目的変数と無関係な乱数列
rfe_data["logp_copy"] = rfe_data["logp"]                # 既存列の複製（冗長）
scaled = pd.DataFrame(StandardScaler().fit_transform(rfe_data), columns=rfe_data.columns, index=rfe_data.index)
rfecv = RFECV(Ridge(alpha=1.0), cv=5, scoring="neg_mean_absolute_error", min_features_to_select=2)
rfecv.fit(scaled, engineered["yield_pct"])
print("元の列数:", scaled.shape[1], "→ 選ばれた列数:", rfecv.n_features_)
pd.DataFrame({"特徴量": rfe_data.columns, "残す": rfecv.support_, "順位": rfecv.ranking_}).sort_values("順位")


### 出力の読み方

- `残す=True`が採用列、`順位=1`が最重要グループ。**わざと混ぜた`noise`（乱数）と`logp_copy`（複製）が削られていれば**、RFECVが「役に立たない列を見抜いて外す」働きをしていると確認できます。
- 木モデル(RandomForest)ではなく線形モデル(Ridge)を使ったのは、**木モデルはノイズ列があっても性能が落ちにくく、RFECVが何も削らないことが多い**ため。**推定器を変えると選択結果も変わる**——特徴量選択も「どの手法で測るか」に依存する、という点も併せて押さえます。
- 選択も交差検証の内側で行うことで、選びすぎ（過学習）を避けています。


## APPENDIX（任意・追加演習）

特徴量づくりの引き出しを増やします。90分の外の自習向けです。まず**交互作用特徴量**——2つの列の
掛け算で「組み合わせの効果」を表します（温度×濃度など）。


In [ ]:
from sklearn.preprocessing import PolynomialFeatures

cols = ["temperature_c", "concentration_m"]
pair = engineered[cols].fillna(engineered[cols].median())
inter = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
out = inter.fit_transform(pair)
display(pd.DataFrame(out, columns=inter.get_feature_names_out(), index=pair.index).head())


### 出力の読み方

元の2列に加え、`temperature_c concentration_m`（掛け算）の列ができます。`interaction_only=True`なので
二乗は作らず組み合わせだけ。「片方が高いときだけもう片方が効く」ような関係を、モデルへ渡せます。


### 連続値を区間に区切る（ビニング）

温度のような連続値を4区間に区切ると、非線形な効果を扱いやすくなったり、解釈しやすくなったりします。
`KBinsDiscretizer`（分位点で等件数に区切る）を使い、区間ごとの平均収率を見ます。


In [ ]:
from sklearn.preprocessing import KBinsDiscretizer

temp = engineered[["temperature_c"]].fillna(engineered["temperature_c"].median())
binner = KBinsDiscretizer(n_bins=4, encode="ordinal", strategy="quantile")
engineered["temp_bin"] = binner.fit_transform(temp).astype(int)
display(engineered.groupby("temp_bin")["yield_pct"].mean().round(1))


### 出力の読み方

区間0（低温）〜3（高温）ごとの平均収率が出ます。中間の区間で収率が高い（山型）なら、第4回で見た
温度の効果と一致。ビニングは効果を見せやすい一方、情報を捨てる面もあるので、元の連続値と併用も検討します。


### 作った特徴量の効き目を、並べ替え重要度で確かめる

第1・12回で使った並べ替え重要度を、この回で作った特徴量を含めた全体に適用します。自作特徴量が
上位に来るかを、holdoutで公平に確認します。


In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

Xe = engineered[added].fillna(engineered[added].median())
Xtr, Xva, ytr, yva = train_test_split(Xe, engineered["yield_pct"], test_size=0.25, random_state=42)
rf = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42).fit(Xtr, ytr)
perm = permutation_importance(rf, Xva, yva, scoring="neg_mean_absolute_error", n_repeats=15, random_state=42)
pd.DataFrame({"特徴量": added, "重要度": perm.importances_mean}).sort_values("重要度", ascending=False).round(3)


### 出力の読み方：3つの見方が食い違うのは正常

ここでは`temperature_distance`が**上位に来ることがあります**。ところが同じ回のCOREでは、相互情報量(MI)で
同じ列が**下位**、アブレーションでは追加しても**MAEがほとんど改善しない**——3つの見方で結論が食い違います。
矛盾ではなく、**それぞれ別の問いに答えているから**です。

- **アブレーション**：その列を入れるか抜くかで最終性能がどう動くか。他の列で代用が効くと、抜いても悪化せず「効果なし」に見える。
- **相互情報量**：その列を単独で見たときの関連の強さ。組み合わせて効く効果は測れない。
- **並べ替え重要度**：学習済みモデルが実際にその列に依存しているか。`temperature_distance`は`temperature_c`から作った相関の強い列なので、モデルがどちらを使うかで重要度が振れやすい。

教訓は2つ。**(1) 1つの指標だけで特徴量の良し悪しを断じない。(2) 元の列と強く相関する派生列（今回の距離特徴量）は、
重要度が不安定になりやすい。** 「作る→交差検証で効果を確かめる→複数の見方で解釈する」という一巡こそが、
思い込みを避ける特徴量設計です。


## よくある誤り

- 意味を説明できない特徴量を大量追加する
- 目的変数由来の値を全データで作って特徴量にする
- 追加前後で分割やモデルも変える

## SELF-STUDY（任意・30〜60分）

- 自作KFold target encodingの有無でMAEを比較する
- RFECVで残った特徴量と、化学的な解釈を突き合わせる

成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

## 振り返りチェック

1. その特徴量はいつ計算できるか
2. target encodingでリークを防ぐ手順は何か
3. 特徴量選択も交差検証の内側で行う理由は何か

答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
